In [1]:
import os, sys, subprocess
 
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
 
install("h2o")

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [3]:
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)
 
print("=" * 60)
print("PART A — FIX PHASE 0 ISSUES")
print("=" * 60)

PART A — FIX PHASE 0 ISSUES


In [4]:
# ──────────────────────────────────────────────────────────────
# A1 — Reload the ORIGINAL clean file from Phase 0
#      (before any encoding was done)
#
# WHY: Your Phase 0 output showed Delivery Status and Order
# Status survived as one-hot encoded columns (features 39-56).
# This happened because Step 9 (encoding) ran BEFORE the
# leakage columns were fully dropped.
#
# Fix: we reload dataco_full_clean.csv and re-apply a stricter
# drop list before encoding. This gives us the correct file.
# ──────────────────────────────────────────────────────────────

In [5]:
FULL_CLEAN_PATH = "D:\d_drive_project\googke solution challange\dataco_full_clean.csv"
 
if not os.path.exists(FULL_CLEAN_PATH):
    raise FileNotFoundError(
        f"Could not find {FULL_CLEAN_PATH}. "
        "Make sure you ran phase0_cleaning.py first."
    )
 
df = pd.read_csv(FULL_CLEAN_PATH)
TARGET = "Late_delivery_risk"
 
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Columns: {list(df.columns)}\n")

Loaded: 180,519 rows × 60 cols
Columns: ['Days for shipping (real)', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Fname', 'Customer Lname', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Order City', 'Order Country', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Profit Per Order', 'Order Region', 'Order State', 'Product Name', 'Product Price', 'order_year', 'order_month', 'order_quarter', 'order_weekday', 'order_day', 'order_is_weekend', 'ship_month', 'ship_weekday', 'ship_quarter', 'delay_gap', 'delay_ratio', 'is_delayed', 'profit_per_unit', 'discount_impact', 'Type_DEBIT', 'Type_PAYMENT', 'Type_TRANSFER', 'Delivery Status_Late delivery', 'Delivery Status_Shipping canceled', 'Delivery Status_Shipping on time', 'Customer Country_Puerto Rico', 'Customer Segment_Corporate', 'Customer Segment_Home Office', 'Market_Europe', 'Market_LATAM', 'Market_Pac

In [6]:
# ──────────────────────────────────────────────────────────────
# A2 — Drop ALL leakage + noise columns that survived Phase 0
#
# WHY each column is dropped:
#
# Delivery Status:
#   Your output showed it survived as one-hot encoded columns:
#   Delivery Status_Late delivery, Shipping canceled, Shipping
#   on time. These directly encode Late_delivery_risk.
#   Model sees "Late delivery" → predicts 1. Fake 99% accuracy.
#
# Order Status:
#   CLOSED, COMPLETE, ON_HOLD, PAYMENT_REVIEW, PENDING,
#   PENDING_PAYMENT, PROCESSING, SUSPECTED_FRAUD all survived
#   as one-hot cols. Correlated with Delivery Status → leakage.
#
# Customer Fname, Customer Lname:
#   782 unique first names, 1109 unique last names.
#   A customer's name has zero relationship to delivery delay.
#   It's near-unique per row → model memorizes names, not learns
#   delivery patterns. Must be removed.
#
# Customer Street:
#   7458 unique values — almost one per customer.
#   The street address is the most granular geo feature and is
#   essentially a unique ID. Model would overfit completely to
#   memorizing which streets had late deliveries. No generalization.
# ──────────────────────────────────────────────────────────────
print("A2 — Dropping leakage + noise columns")
print("-" * 50)
 
# Check which of these exist (some may have already been one-hot
# encoded into sub-columns by Phase 0 Step 9)
EXTRA_DROP = [
    # Leakage — directly encodes the target
    "Delivery Status",
    "Order Status",
    # One-hot encoded versions of leakage (if they exist)
    "Delivery Status_Late delivery",
    "Delivery Status_Shipping canceled",
    "Delivery Status_Shipping on time",
    "Delivery Status_Advance shipping",
    "Order Status_CLOSED",
    "Order Status_COMPLETE",
    "Order Status_ON_HOLD",
    "Order Status_PAYMENT_REVIEW",
    "Order Status_PENDING",
    "Order Status_PENDING_PAYMENT",
    "Order Status_PROCESSING",
    "Order Status_SUSPECTED_FRAUD",
    # Noise — near-unique identifiers with no predictive signal
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
]
 
dropped = [c for c in EXTRA_DROP if c in df.columns]
not_found = [c for c in EXTRA_DROP if c not in df.columns]
 
print(f"Dropping {len(dropped)} leakage/noise columns:")
for c in dropped:
    print(f"  ✗ {c}")
if not_found:
    print(f"\nNot present (already gone or never existed): {len(not_found)} cols")
 
df.drop(columns=dropped, inplace=True)
print(f"\n✓ Shape after fix: {df.shape[0]:,} × {df.shape[1]}")

A2 — Dropping leakage + noise columns
--------------------------------------------------
Dropping 14 leakage/noise columns:
  ✗ Delivery Status_Late delivery
  ✗ Delivery Status_Shipping canceled
  ✗ Delivery Status_Shipping on time
  ✗ Order Status_CLOSED
  ✗ Order Status_COMPLETE
  ✗ Order Status_ON_HOLD
  ✗ Order Status_PAYMENT_REVIEW
  ✗ Order Status_PENDING
  ✗ Order Status_PENDING_PAYMENT
  ✗ Order Status_PROCESSING
  ✗ Order Status_SUSPECTED_FRAUD
  ✗ Customer Fname
  ✗ Customer Lname
  ✗ Customer Street

Not present (already gone or never existed): 3 cols

✓ Shape after fix: 180,519 × 46


In [7]:
# ──────────────────────────────────────────────────────────────
# A3 — Verify no leakage remains
#
# WHY: Before sending to AutoML, confirm with a correlation
# check that no remaining column has r > 0.95 with the target.
# A legitimate feature should never correlate that perfectly —
# if it does, it's almost certainly a form of leakage.
# ──────────────────────────────────────────────────────────────
print("\nA3 — Leakage correlation check")
print("-" * 50)
 
# Only check numeric columns
num_cols = df.select_dtypes(include=["int64","float64","int32",
                                      "float32","uint8","bool"]).columns
correlations = df[num_cols].corrwith(df[TARGET]).abs().sort_values(ascending=False)
 
print("Top 15 correlations with target:")
print(correlations.head(15).to_string())
 
suspicious = correlations[correlations > 0.95].drop(TARGET, errors="ignore")
if len(suspicious) > 0:
    print(f"\n⚠ SUSPICIOUS (r > 0.95): {list(suspicious.index)}")
    print("  These may still be leakage — review before proceeding.")
else:
    print("\n✓ No suspicious correlations (all r < 0.95) — safe to proceed")


A3 — Leakage correlation check
--------------------------------------------------
Top 15 correlations with target:
Late_delivery_risk              1.0000
is_delayed                      0.9515
delay_gap                       0.8273
delay_ratio                     0.7267
Shipping Mode_Standard Class    0.4097
Days for shipping (real)        0.4014
Days for shipment (scheduled)   0.3694
Shipping Mode_Second Class      0.2157
Type_TRANSFER                   0.0780
Shipping Mode_Same Day          0.0436
Type_DEBIT                      0.0379
Type_PAYMENT                    0.0297
Order Region                    0.0062
Market_LATAM                    0.0060
Customer City                   0.0051

⚠ SUSPICIOUS (r > 0.95): ['is_delayed']
  These may still be leakage — review before proceeding.


In [8]:
# ──────────────────────────────────────────────────────────────
# A4 — Re-encode any remaining categoricals cleanly
#
# WHY: After dropping leakage columns, we re-check for any
# remaining object columns and encode them. This ensures the
# dataframe is fully numeric before H2O receives it.
# Note: H2O can handle categoricals natively, but we encode
# here to be safe across all AutoML frameworks.
# ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
 
print("\nA4 — Re-encoding remaining categoricals")
print("-" * 50)
 
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
if TARGET in cat_cols:
    cat_cols.remove(TARGET)
 
if not cat_cols:
    print("✓ No categorical columns remaining — all numeric")
else:
    LOW_CARD = 10
    low_c  = [c for c in cat_cols if df[c].nunique() < LOW_CARD]
    high_c = [c for c in cat_cols if df[c].nunique() >= LOW_CARD]
 
    if low_c:
        df = pd.get_dummies(df, columns=low_c, drop_first=True, dtype=int)
        print(f"  One-hot encoded: {low_c}")
 
    le = LabelEncoder()
    for col in high_c:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))
            print(f"  Label encoded : {col} ({df[col].nunique()} classes)")
 
print(f"\n✓ Final clean shape: {df.shape[0]:,} × {df.shape[1]}")


A4 — Re-encoding remaining categoricals
--------------------------------------------------
✓ No categorical columns remaining — all numeric

✓ Final clean shape: 180,519 × 46


In [9]:
# Print final feature list
features = [c for c in df.columns if c != TARGET]
print(f"\nFinal features for AutoML ({len(features)}):")
for i, col in enumerate(features, 1):
    corr_with_target = abs(df[col].corr(df[TARGET]))
    print(f"  {i:02d}. {col:45s} (r={corr_with_target:.3f})")


Final features for AutoML (45):
  01. Days for shipping (real)                      (r=0.401)
  02. Days for shipment (scheduled)                 (r=0.369)
  03. Category Name                                 (r=0.001)
  04. Customer City                                 (r=0.005)
  05. Customer State                                (r=0.002)
  06. Customer Zipcode                              (r=0.003)
  07. Order City                                    (r=0.004)
  08. Order Country                                 (r=0.002)
  09. Order Item Discount Rate                      (r=0.000)
  10. Order Item Product Price                      (r=0.001)
  11. Order Item Profit Ratio                       (r=0.001)
  12. Order Item Quantity                           (r=0.000)
  13. Sales                                         (r=0.002)
  14. Order Profit Per Order                        (r=0.003)
  15. Order Region                                  (r=0.006)
  16. Order State                    

In [10]:
# ──────────────────────────────────────────────────────────────
# A5 — Rebuild time-based train/test split on fixed data
#
# WHY: After dropping columns and re-encoding, we rebuild the
# split from scratch on the fixed dataframe. We sort by the
# temporal features (order_year, order_month) so the earliest
# 80% becomes train and the latest 20% becomes test.
# ──────────────────────────────────────────────────────────────

print("-" * 50)
 
sort_cols = [c for c in ["order_year","order_month","order_day"] if c in df.columns]
df_sorted = df.sort_values(sort_cols).reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.80)
 
train_df = df_sorted.iloc[:split_idx].copy()
test_df  = df_sorted.iloc[split_idx:].copy()
 
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]
 
print(f"✓ Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows")
print(f"  Train target: 0={( y_train==0).mean()*100:.1f}%  1={(y_train==1).mean()*100:.1f}%")
print(f"  Test  target: 0={(  y_test==0).mean()*100:.1f}%  1={(y_test==1).mean()*100:.1f}%")

--------------------------------------------------
✓ Train: 144,415 rows  |  Test: 36,104 rows
  Train target: 0=45.1%  1=54.9%
  Test  target: 0=45.3%  1=54.7%


In [12]:
# # Save the FIXED clean files
# train_df.to_csv("/content/dataco_train_FIXED.csv", index=False)
# test_df.to_csv("/content/dataco_test_FIXED.csv",   index=False)
# print("\n✓ Saved fixed files:")
# print("  /content/dataco_train_FIXED.csv  ← use this for AutoML")
# print("  /content/dataco_test_FIXED.csv   ← hold out for evaluation")

In [14]:
# ================================================================
print("\n" + "=" * 60)
print("PART B — H2O AutoML TRAINING")
print("=" * 60)
# ================================================================

import h2o
from h2o.automl import H2OAutoML


PART B — H2O AutoML TRAINING


In [15]:
# ──────────────────────────────────────────────────────────────
# B1 — Start H2O cluster
#
# WHY: H2O runs as an in-process Java server. We initialize it
# with max_mem_size to give it enough RAM for 180k rows.
# In Colab, the runtime has ~12 GB RAM — we give H2O 8 GB.
# nthreads=-1 means use all available CPU cores.
# ──────────────────────────────────────────────────────────────
print("B1 — Starting H2O cluster")
print("-" * 50)
 
h2o.init(
    max_mem_size="8G",
    nthreads=-1,
    verbose=False
)
print(f"✓ H2O version: {h2o.__version__}")
print(f"  Cluster: {h2o.cluster().cloud_name}")

B1 — Starting H2O cluster
--------------------------------------------------
✓ H2O version: 3.46.0.10
  Cluster: H2O_from_python_satya_hj9tpi


In [16]:
# ──────────────────────────────────────────────────────────────
# B2 — Convert pandas DataFrames to H2O frames
#
# WHY: H2O has its own distributed data format (H2OFrame).
# We convert our pandas DataFrames using h2o.H2OFrame().
# The target column must be a "factor" (categorical) for
# classification — H2O treats numeric targets as regression.
# ──────────────────────────────────────────────────────────────

print("\nB2 — Converting data to H2O frames")
print("-" * 50)

h2o_train = h2o.H2OFrame(train_df)
h2o_test  = h2o.H2OFrame(test_df)

# Convert target to factor (classification, not regression)
h2o_train[TARGET] = h2o_train[TARGET].asfactor()
h2o_test[TARGET]  = h2o_test[TARGET].asfactor()
 
print(f"✓ H2O train frame: {h2o_train.shape[0]:,} rows × {h2o_train.shape[1]} cols")
print(f"✓ H2O test  frame: {h2o_test.shape[0]:,} rows × {h2o_test.shape[1]} cols")
print(f"  Target levels: {h2o_train[TARGET].levels()}")





B2 — Converting data to H2O frames
--------------------------------------------------
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
✓ H2O train frame: 144,415 rows × 46 cols
✓ H2O test  frame: 36,104 rows × 46 cols
  Target levels: [['0', '1']]
